# RXN Sandbox

### Celery setup and helper functions

In [1]:
# All necessary imports
import time
import base64
import pandas as pd
import ipywidgets as widgets
from io import BytesIO
from rdkit import Chem
from rdkit.Chem import Draw
from celery import Celery
from celery.result import AsyncResult
from typing import Optional, Dict, Any
from IPython.display import display, HTML, clear_output, Markdown


# Initialize Celery
celery_app = Celery()
#print("Broker:", celery_app.conf.broker_url)
#print("Backend:", celery_app.conf.result_backend)


# Wait for the celery result
def wait_for_result(app: Celery, task_id: str, timeout: float = 120.0, poll: float = 0.5) -> Dict[str, Any]:
    """
    Poll for a result with a timeout. If task updates state with meta (e.g., PROGRESS),
    we surface that along the way.
    """
    res = AsyncResult(task_id, app=app)
    t0 = time.time()
    last_state = None

    while True:
        state = res.state
        if state != last_state:
            print(f"State: {state} | Info: {res.info}")
            last_state = state

        if res.ready():
            # could be SUCCESS or FAILURE; .get() will raise on FAILURE
            return res.get(propagate=False)  # returns exception object if failed

        if time.time() - t0 > timeout:
            raise TimeoutError(f"Task {task_id} did not finish in {timeout} seconds.")
        time.sleep(poll)


# Build results dataframe
def build_results_dataframe(records):
    """
    Convert Celery result(s) into a DataFrame.
    Only keeps: smiles, predicted_smiles, confidence
    """
    if records is None:
        return pd.DataFrame()
    if isinstance(records, dict):
        records = [records]
    if not isinstance(records, list):
        return pd.DataFrame()

    df = pd.DataFrame(records)
    
    if {"reactants", "product"}.issubset(df.columns):
        df.insert(
            0,  # put as first column (or change to another position)
            "Predicted Reaction",
            df["reactants"].fillna("").astype(str) + " → " + df["product"].fillna("").astype(str)
        )
        # Optionally remove the original columns:
        df = df.drop(columns=["reactants", "product"])

    cols = []
    for c in ["Predicted Reaction", "confidence"]:
        if c in df.columns:
            cols.append(c)

    return df[cols]


# Show results table
def style_results_table(df, caption="Reaction Predictions", align="left"):
    """
    Style the table:
    - Bigger, bold caption
    - Confidence mapped red→yellow→green (0→1), fixed range vmin=0, vmax=1
    - Removes index
    """

    clear_output(wait=True)
    
    if df.empty:
        display(HTML("<p><em>No results to display.</em></p>"))
        return

    df = df.copy()

    # Build the Styler
    styler = df.style

    # Confidence formatting + red→yellow→green gradient (low→high)
    # Lock range to [0, 1] so colors are consistent even for partial data.
    if "confidence" in df.columns:
        try:
            styler = styler.format({"confidence": "{:.3f}"}, escape=None)
        except TypeError:
            styler = styler.format({"confidence": "{:.3f}"}, escape=False)
        styler = styler.background_gradient(
            subset=["confidence"],
            cmap="RdYlGn",   # red (low) → yellow → green (high)
            vmin=0.0,
            vmax=1.0,
        )
        styler = styler.set_properties(
            subset=["confidence"], **{"font-weight": "500",})

    # SMILES styling: bold + monospace + wrapping
    if "Predicted Reaction" in df.columns:
        styler = styler.set_properties(
            subset=["Predicted Reaction"], 
            **{
                "background-color": "#fafafa", 
                "text-align": align,
                "font-weight": "600",
                "font-family": "ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace",
            })

    # Table styles (caption, headers, borders)
    styler = styler.set_table_styles(
        [
            {
                "selector": "caption",
                "props": [
                    ("caption-side", "top"),
                    ("font-weight", "700"),
                    ("font-size", "1.2rem"),
                    ("margin-bottom", "10px"),
                ],
            },
            {
                "selector": "th.col_heading",
                "props": [
                    ("font-weight", "700"),
                    ("padding", "8px"),
                    ("text-align", "center"),
                ],
            },
            {
                "selector": "td",
                "props": [
                    ("padding", "8px"),
                    ("vertical-align", "center"),
                ],
            },
            {
                "selector": "table",
                "props": [
                    ("border-collapse", "separate"),
                    ("border-spacing", "0px"),
                    ("width", "100%"),
                ],
            },
        ]
    )

    # Render with inline HTML (formatters already handled escaping logic)
    try:
        styler = styler.hide(axis="index")
    except Exception:
        styler = styler.hide_index()

    # Use modern API first; fall back for older pandas
    try:
        html = styler.set_caption(caption).to_html()  # escape behavior controlled in format()
    except TypeError:
        html = styler.set_caption(caption).to_html(escape=False)

    display(HTML(html))


# Display a retrosynthesis tree textually
def display_route(route, idx):
    display(Markdown(
        f"## Route {idx + 1}\n"
        f"- **Score:** `{route['score']:.3f}`\n"
        f"- **Steps:** `{route['steps']}`"
    ))

    open_nodes = route.get("open_nodes", [])
    if open_nodes:
        display(Markdown(f"**Remaining molecules:** `{', '.join(open_nodes)}`"))

    for i, arc in enumerate(route.get("arcs", []), start=1):
        product = arc["product"]
        reactants = ".".join(arc["reactants"])

        display(Markdown(
            f"#### Step {i}\n"
            f"**Product:** `{product}`  \n"
            f"**Reactants:** `{reactants}`  \n"
            f"- Forward likelihood: `{arc['forward_likelihood']:.3f}`  \n"
            f"- Arc score: `{arc['arc_score']:.3f}`  \n"
            f"- Retro confidence: `{arc.get('confidence', 'n/a')}`"
        ))




# Convert a molecule to PNG for faster representation
def mol_png_base64(smiles, size=(250, 200)):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return ""

    img = Draw.MolToImage(mol, size=size)
    buf = BytesIO()
    img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode("utf-8")


# Draw a row of reactants that make a retrosynthesis tree step
def draw_reactant_row_png(reactants, expanded_smiles=None):
    imgs = [mol_png_base64(smi) for smi in reactants]

    html = """
    <div style="
        display: flex;
        flex-direction: row;
        justify-content: flex-start;
        align-items: flex-start;
        gap: 20px;
        overflow-x: auto;
        padding: 12px 20px;
        border: 1px solid #ddd;
        margin-bottom: 16px;
        box-sizing: border-box;
    ">
    """

    for smi, img in zip(reactants, imgs):
        is_expanded = (smi == expanded_smiles)

        border = "2px solid #1f77b4" if is_expanded else "1px solid #ccc"
        label = "<div style='font-weight:700; color:#1f77b4;'>expanded ↓</div>" if is_expanded else ""

        html += f"""
        <div style="
            display: flex;
            flex-direction: column;
            align-items: center;
            flex: 0 0 auto;
            min-width: 260px;
            padding: 6px;
            border: {border};
            border-radius: 6px;
        ">
            <img src="data:image/png;base64,{img}" style="display:block;"/>
            <div style="
                margin-top: 6px;
                font-size: 14px;
                font-weight: 700;
                text-align: center;
                word-break: break-all;
                font-family: monospace;
            ">
                {smi}
            </div>
            {label}
        </div>
        """

    html += "</div>"
    display(HTML(html))


# Draw the full retrosynthesis tree
def draw_route_png(route):
    display(Markdown("## Target molecule"))
    draw_reactant_row_png([route["arcs"][0]["product"]])

    for i, arc in enumerate(route["arcs"]):
        display(Markdown(f"## Step {i + 1} reactants"))

        # Determine which reactant is further decomposed
        expanded = None
        if i + 1 < len(route["arcs"]):
            expanded = route["arcs"][i + 1]["product"]

        draw_reactant_row_png(
            arc["reactants"],
            expanded_smiles=expanded
        )


# Builds a selector to display the full retrosynthesis tree for the selected route
def tree_route_selector(results):
    options = [
        (f"Route {i+1} | score={r['score']:.3f}", i)
        for i, r in enumerate(results)
    ]
    
    def show_route(idx):
        draw_route_png(results[idx])
    
    dropdown = widgets.Dropdown(
        options=options,
        description="Route",
    )
    
    ui = widgets.interactive(show_route, idx=dropdown)
    display(ui)

### Product prediction for a batch of reactions

In [5]:
# Set up a list of reactants to make predictions
reactants_list = ["CCI.O=Cc1ccc([N+](=O)[O-])c(O)c1", "CCOc1cc(O)c(C=O)cc1OCC.OCCCBr", "C=CCc1cc(OCc2ccccc2)ccc1O.CCBr"]

# Setup task kwargs
kwargs = {
    "topn": 1,        # Number of results per reactant
    "num_beams": 3,   # Number of beams used for prediction. Must be >= topn
    "device": None,   # Device used for predicting, either "cuda" or "cpu", None defaults to cuda if available
    "ckpt_forward": "Pistachio2025Q2-Forward",  # Default forward model
    "vocab": "Pistachio2025Q2",         # Vocab for default forward model
    # "ckpt_forward_path": "models/forward/Pistachio2025Q2-Forward.ckpt",  # Can be used instead of ckpt_forward
    # "vocab_path": "vocab/Pistachio2025Q2.txt",  # Can be used instead of vocab
}

# Send the product_prediction task with the reaction list and kwargs
task = celery_app.send_task(
    "tasks.product_prediction",
    [reactants_list],
    kwargs=kwargs,
    queue="product_prediction",
)
print("Task sent. Assigned task_id: {}".format(task.id))

# Use the task id to get the result. Increase timeout if needed.
response = wait_for_result(celery_app, task.id, timeout=180)

Task sent. Assigned task_id: 31bccfad-38b8-4cc5-965f-08552ed376f6
State: PENDING | Info: None
State: STARTED | Info: None
State: SUCCESS | Info: {'result': [{'reactants': 'CCI.O=Cc1ccc([N+](=O)[O-])c(O)c1', 'product': 'CCOc1cc(C=O)ccc1[N+](=O)[O-]', 'confidence': 0.9987357632971271, 'smiles': 'CCI.O=Cc1ccc([N+](=O)[O-])c(O)c1>>CCOc1cc(C=O)ccc1[N+](=O)[O-]'}, {'reactants': 'CCOc1cc(O)c(C=O)cc1OCC.OCCCBr', 'product': 'CCOc1cc(C=O)c(OCCCO)cc1OCC', 'confidence': 0.992653825419336, 'smiles': 'CCOc1cc(O)c(C=O)cc1OCC.OCCCBr>>CCOc1cc(C=O)c(OCCCO)cc1OCC'}, {'reactants': 'C=CCc1cc(OCc2ccccc2)ccc1O.CCBr', 'product': 'C=CCc1cc(OCc2ccccc2)ccc1OCC', 'confidence': 0.9966198460612675, 'smiles': 'C=CCc1cc(OCc2ccccc2)ccc1O.CCBr>>C=CCc1cc(OCc2ccccc2)ccc1OCC'}], 'time': 3.18107008934021}


In [6]:
# Build a dataframe with the result in the response
df = build_results_dataframe(response["result"])

# Create and show the data in a table
style_results_table(df, caption="Product Prediction Results", align="center")

Predicted Reaction,confidence
CCI.O=Cc1ccc([N+](=O)[O-])c(O)c1 → CCOc1cc(C=O)ccc1[N+](=O)[O-],0.999
CCOc1cc(O)c(C=O)cc1OCC.OCCCBr → CCOc1cc(C=O)c(OCCCO)cc1OCC,0.993
C=CCc1cc(OCc2ccccc2)ccc1O.CCBr → C=CCc1cc(OCc2ccccc2)ccc1OCC,0.997


### Product prediction for a single reaction - top 3 results

In [7]:
# Set up a list of reactants to make predictions
reactants_list = ["CCI.O=Cc1ccc([N+](=O)[O-])c(O)c1"]

# Setup task kwargs
kwargs = {
    "topn": 3,        # Number of results per reactant
    "num_beams": 5,   # Number of beams used for prediction. Must be >= topn
    "device": None,   # Device used for predicting, either "cuda" or "cpu", None defaults to cuda if available
    "ckpt_forward": "Pistachio2025Q2-Forward",  # Default forward model
    "vocab": "Pistachio2025Q2",         # Vocab for default forward model
    # "ckpt_forward_path": "models/forward/Pistachio2025Q2-Forward.ckpt",  # Can be used instead of ckpt_forward
    # "vocab_path": "vocab/Pistachio2025Q2.txt",  # Can be used instead of vocab
}

# Send the product_prediction task with the reaction list and kwargs
task = celery_app.send_task(
    "tasks.product_prediction",
    [reactants_list],
    kwargs=kwargs,
    queue="product_prediction",
)
print("Task sent. Assigned task_id: {}".format(task.id))

# Use the task id to get the result. Increase timeout if needed.
response = wait_for_result(celery_app, task.id, timeout=180)

Task sent. Assigned task_id: 10bdf9f8-e49f-4c5f-8f11-5d03f3f75d83
State: PENDING | Info: None
State: STARTED | Info: None
State: SUCCESS | Info: {'result': [{'reactants': 'CCI.O=Cc1ccc([N+](=O)[O-])c(O)c1', 'product': 'CCOc1cc(C=O)ccc1[N+](=O)[O-]', 'confidence': 0.9987357632971271, 'smiles': 'CCI.O=Cc1ccc([N+](=O)[O-])c(O)c1>>CCOc1cc(C=O)ccc1[N+](=O)[O-]'}, {'reactants': 'CCI.O=Cc1ccc([N+](=O)[O-])c(O)c1', 'product': 'CCOc1cc(C=O)ccc1[N+](=O)[O-].CCOc1cc(C=O)ccc1[N+](=O)[O-]', 'confidence': 0.8466943868002893, 'smiles': 'CCI.O=Cc1ccc([N+](=O)[O-])c(O)c1>>CCOc1cc(C=O)ccc1[N+](=O)[O-].CCOc1cc(C=O)ccc1[N+](=O)[O-]'}, {'reactants': 'CCI.O=Cc1ccc([N+](=O)[O-])c(O)c1', 'product': 'CCOc1cc(C=O)ccc1[N+](=O)[O-].CCOc1cc(CO)ccc1[N+](=O)[O-]', 'confidence': 0.7820610951050019, 'smiles': 'CCI.O=Cc1ccc([N+](=O)[O-])c(O)c1>>CCOc1cc(C=O)ccc1[N+](=O)[O-].CCOc1cc(CO)ccc1[N+](=O)[O-]'}], 'time': 3.020906448364258}


In [8]:
# Build a dataframe with the result in the response
df = build_results_dataframe(response["result"])

# Create and show the data in a table
style_results_table(df, caption="Product Prediction Results")

Predicted Reaction,confidence
CCI.O=Cc1ccc([N+](=O)[O-])c(O)c1 → CCOc1cc(C=O)ccc1[N+](=O)[O-],0.999
CCI.O=Cc1ccc([N+](=O)[O-])c(O)c1 → CCOc1cc(C=O)ccc1[N+](=O)[O-].CCOc1cc(C=O)ccc1[N+](=O)[O-],0.847
CCI.O=Cc1ccc([N+](=O)[O-])c(O)c1 → CCOc1cc(C=O)ccc1[N+](=O)[O-].CCOc1cc(CO)ccc1[N+](=O)[O-],0.782


### Retrosynthesis prediction

In [9]:
# Choose product for retrosynthesis prediction
product = "C=CC(=C)C[Si](C)(C)C"

# Setup task kwargs
kwargs = {
    "topn": 15,       # Number of results per reactant
    "num_beams": 15,  # Number of beams used for prediction. Must be >= topn
    "fap": 0.6,       # Forward likelihood acceptance probability (not length averaged)
    "fld": 0.2,       # Forward likelihood delta required between the top2 forward prediction results
    "device": None,   # Device used for predicting, either "cuda" or "cpu", None defaults to cuda if available
    "ckpt_forward": "Pistachio2025Q2-Forward",  # Default forward model
    "ckpt_retro": "Pistachio2025Q2-Retro",    # Default retrosynthesis model
    "vocab": "Pistachio2025Q2",         # Vocab for default forward and retrosynthesis models
    # "ckpt_forward_path": "models/forward/Pistachio2025Q2-Forward.ckpt",  # Can be used instead of ckpt_forward
    # "ckpt_retro_path": "models/retrosynthesis/Pistachio2025Q2-Retro.ckpt",  # Can be used instead of ckpt_retro
    # "vocab_path": "vocab/Pistachio2025Q2.txt",  # Can be used instead of vocab
}

# Send the retro_prediction task with the product and kwargs
task = celery_app.send_task(
    "tasks.retro_prediction",
    [product],
    kwargs=kwargs,
    queue="retro_prediction",
)
print("Task sent. Assigned task_id: {}".format(task.id))

# Use the task id to get the result. Increase timeout if needed.
response = wait_for_result(celery_app, task.id, timeout=300)

Task sent. Assigned task_id: 2ac795d1-67f2-42ab-b6e8-db9d0f04e37d
State: PENDING | Info: None
State: STARTED | Info: None
State: SUCCESS | Info: {'result': [{'reactants': 'C1CCOC1.C=CC(=C)Cl.C[Si](C)(C)[CH2][Mg][Cl]', 'product': 'C=CC(=C)C[Si](C)(C)C', 'confidence': 0.8633497322540727, 'smiles': 'C1CCOC1.C=CC(=C)Cl.C[Si](C)(C)[CH2][Mg][Cl]>>C=CC(=C)C[Si](C)(C)C'}, {'reactants': 'C1CCOC1.C=CC(=C)Cl.C[Si](C)(C)[CH2][Mg][Cl].N#N', 'product': 'C=CC(=C)C[Si](C)(C)C', 'confidence': 0.853555662500899, 'smiles': 'C1CCOC1.C=CC(=C)Cl.C[Si](C)(C)[CH2][Mg][Cl].N#N>>C=CC(=C)C[Si](C)(C)C'}, {'reactants': 'C1CCOC1.C=CC(=C)Cl.CCOCC.C[Si](C)(C)[CH2][Mg][Cl]', 'product': 'C=CC(=C)C[Si](C)(C)C', 'confidence': 0.8437746350333076, 'smiles': 'C1CCOC1.C=CC(=C)Cl.CCOCC.C[Si](C)(C)[CH2][Mg][Cl]>>C=CC(=C)C[Si](C)(C)C'}, {'reactants': 'C1CCOC1.C=CC(=C)Cl.C[Si](C)(C)[CH2][Mg][Cl].O.[Cl-].[NH4+]', 'product': 'C=CC(=C)C[Si](C)(C)C', 'confidence': 0.8341045215463807, 'smiles': 'C1CCOC1.C=CC(=C)Cl.C[Si](C)(C)[CH2][Mg

In [10]:
# Build a dataframe with the result in the response
df = build_results_dataframe(response["result"])

# Create and show the data in a table
style_results_table(df, caption="Retrosynthesis Prediction Results", align="right")

Predicted Reaction,confidence
C1CCOC1.C=CC(=C)Cl.C[Si](C)(C)[CH2][Mg][Cl] → C=CC(=C)C[Si](C)(C)C,0.863
C1CCOC1.C=CC(=C)Cl.C[Si](C)(C)[CH2][Mg][Cl].N#N → C=CC(=C)C[Si](C)(C)C,0.854
C1CCOC1.C=CC(=C)Cl.CCOCC.C[Si](C)(C)[CH2][Mg][Cl] → C=CC(=C)C[Si](C)(C)C,0.844
C1CCOC1.C=CC(=C)Cl.C[Si](C)(C)[CH2][Mg][Cl].O.[Cl-].[NH4+] → C=CC(=C)C[Si](C)(C)C,0.834
C1CCOC1.C=CC(=C)Cl.CCOCC.C[Si](C)(C)[CH2][Mg][Cl].N#N → C=CC(=C)C[Si](C)(C)C,0.834
C1CCOC1.C=CC(=C)Cl.CCOCC.C[Si](C)(C)[CH2][Mg][Cl].O.[Cl-].[NH4+] → C=CC(=C)C[Si](C)(C)C,0.833
C1CCOC1.C=CC(=C)Cl.CCOCC.C[Si](C)(C)[CH2][Mg][Cl].Cl.O → C=CC(=C)C[Si](C)(C)C,0.830
C1CCOC1.C=CC(=C)Cl.CCOCC.C[Si](C)(C)[CH2][Mg][Cl].O.[Cl-].[Mg].[NH4+] → C=CC(=C)C[Si](C)(C)C,0.830


### Retro tree prediction

In [11]:
# Choose product for retrosynthesis tree prediction
product = "C1C(C[Si](C)(C)C)=CCC2C(=O)OC(=O)C12"
#product = "Cc1cc2scnc2cc1N"
#product = "CC(C)(C(=O)O)C1C=CC=C(C2CC2)C1=O"
#product = "Nc1ccc2scnc2c1Br"

# Setup task kwargs
kwargs = {
    "topn": 15,       # Number of results per reactant
    "num_beams": 15,  # Number of beams used for prediction. Must be >= topn
    "fap": 0.6,       # Forward likelihood acceptance probability (not length averaged)
    "fld": 0.2,       # Forward likelihood delta required between the top2 forward prediction results
    "max_depth": 4,   # Max depth of the retrosynthesis tree
    "beam_width": 6,  # Max amount of nodes being expanded in each step
    "device": None,   # Device used for predicting, either "cuda" or "cpu", None defaults to cuda if available
    "ckpt_forward": "Pistachio2025Q2-Forward",  # Default forward model
    "ckpt_retro": "Pistachio2025Q2-Retro",    # Default retrosynthesis model
    "vocab": "Pistachio2025Q2",         # Vocab for default forward and retrosynthesis models
    # "ckpt_forward_path": "models/forward/Pistachio2025Q2-Forward.ckpt",  # Can be used instead of ckpt_forward
    # "ckpt_retro_path": "models/retrosynthesis/Pistachio2025Q2-Retro.ckpt",  # Can be used instead of ckpt_retro
    # "vocab_path": "vocab/Pistachio2025Q2.txt",  # Can be used instead of vocab
}

# Send the retro_prediction_tree task with the product and kwargs
task = celery_app.send_task(
    "tasks.retro_prediction_tree",
    [product],
    kwargs=kwargs,
    queue="retro_prediction",
)
print("Task sent. Assigned task_id: {}".format(task.id))

# Use the task id to get the result. Increase timeout if needed.
response = wait_for_result(celery_app, task.id, timeout=3000)

Task sent. Assigned task_id: 26ea63e0-2527-4baf-9905-673f5d98442e
State: PENDING | Info: None
State: STARTED | Info: None
State: SUCCESS | Info: {'result': [{'open_nodes': ['ClCCl', '[Cl][Ru]([Cl])(=[CH]c1ccccc1)([P](C1CCCCC1)(C1CCCCC1)C1CCCCC1)[P](C1CCCCC1)(C1CCCCC1)C1CCCCC1', 'C=CCC1C(=O)OC(=O)C1CC(=C)C[Si](C)(C)C'], 'expanded': ['C[Si](C)(C)CC1=CCC2C(=O)OC(=O)C2C1'], 'score': 0.7056475777714836, 'steps': 1, 'arcs': [{'product': 'C[Si](C)(C)CC1=CCC2C(=O)OC(=O)C2C1', 'reactants': ['C=CCC1C(=O)OC(=O)C1CC(=C)C[Si](C)(C)C', 'ClCCl', '[Cl][Ru]([Cl])(=[CH]c1ccccc1)([P](C1CCCCC1)(C1CCCCC1)C1CCCCC1)[P](C1CCCCC1)(C1CCCCC1)C1CCCCC1'], 'forward_likelihood': 0.8842267990112305, 'arc_score': 0.7056475777704836, 'confidence': 0.9439050974158504, 'metadata': {'reactants': 'C=CCC1C(=O)OC(=O)C1CC(=C)C[Si](C)(C)C.ClCCl.[Cl][Ru]([Cl])(=[CH]c1ccccc1)([P](C1CCCCC1)(C1CCCCC1)C1CCCCC1)[P](C1CCCCC1)(C1CCCCC1)C1CCCCC1', 'product': 'C[Si](C)(C)CC1=CCC2C(=O)OC(=O)C2C1', 'confidence': 0.9439050974158504, 'smile

In [12]:
# Textual retrosynthesis tree representation
for i, route in enumerate(response["result"]):
    display_route(route, i)

## Route 1
- **Score:** `0.706`
- **Steps:** `1`

**Remaining molecules:** `ClCCl, [Cl][Ru]([Cl])(=[CH]c1ccccc1)([P](C1CCCCC1)(C1CCCCC1)C1CCCCC1)[P](C1CCCCC1)(C1CCCCC1)C1CCCCC1, C=CCC1C(=O)OC(=O)C1CC(=C)C[Si](C)(C)C`

#### Step 1
**Product:** `C[Si](C)(C)CC1=CCC2C(=O)OC(=O)C2C1`  
**Reactants:** `C=CCC1C(=O)OC(=O)C1CC(=C)C[Si](C)(C)C.ClCCl.[Cl][Ru]([Cl])(=[CH]c1ccccc1)([P](C1CCCCC1)(C1CCCCC1)C1CCCCC1)[P](C1CCCCC1)(C1CCCCC1)C1CCCCC1`  
- Forward likelihood: `0.884`  
- Arc score: `0.706`  
- Retro confidence: `0.9439050974158504`

## Route 2
- **Score:** `0.371`
- **Steps:** `1`

**Remaining molecules:** `ClCCl, Cc1cc(C)c(N2CCN(c3c(C)cc(C)cc3C)[C]2=[Ru]([Cl])([Cl])(=[CH]c2ccccc2)[P](C2CCCCC2)(C2CCCCC2)C2CCCCC2)c(C)c1, C=CCC1C(=O)OC(=O)C1CC(=C)C[Si](C)(C)C`

#### Step 1
**Product:** `C[Si](C)(C)CC1=CCC2C(=O)OC(=O)C2C1`  
**Reactants:** `C=CCC1C(=O)OC(=O)C1CC(=C)C[Si](C)(C)C.Cc1cc(C)c(N2CCN(c3c(C)cc(C)cc3C)[C]2=[Ru]([Cl])([Cl])(=[CH]c2ccccc2)[P](C2CCCCC2)(C2CCCCC2)C2CCCCC2)c(C)c1.ClCCl`  
- Forward likelihood: `0.809`  
- Arc score: `0.371`  
- Retro confidence: `0.9503329457084253`

... (32 outputs left out)

**Remaining molecules:** `O=C1C=CC(=O)O1, [NH4+], C=CC(=C)O, [Cl-], O=S(Cl)Cl, Cc1ccccc1, C[Si](C)(C)[CH2][Mg][Cl], C1CCOC1, O`

#### Step 1
**Product:** `C[Si](C)(C)CC1=CCC2C(=O)OC(=O)C2C1`  
**Reactants:** `C=CC(=C)C[Si](C)(C)C.Cc1ccccc1.O=C1C=CC(=O)O1`  
- Forward likelihood: `0.393`  
- Arc score: `0.437`  
- Retro confidence: `0.9271861522313083`

#### Step 2
**Product:** `C=CC(=C)C[Si](C)(C)C`  
**Reactants:** `C1CCOC1.C=CC(=C)Cl.C[Si](C)(C)[CH2][Mg][Cl].O.[Cl-].[NH4+]`  
- Forward likelihood: `0.917`  
- Arc score: `1.025`  
- Retro confidence: `0.8341045215463807`

#### Step 3
**Product:** `C=CC(=C)Cl`  
**Reactants:** `C=CC(=C)O.O=S(Cl)Cl`  
- Forward likelihood: `0.404`  
- Arc score: `0.393`  
- Retro confidence: `0.8269444351225532`

In [13]:
# Graphical retrosynthesis tree representation with route selection
tree_route_selector(response["result"])

interactive(children=(Dropdown(description='Route', options=(('Route 1 | score=0.706', 0), ('Route 2 | score=0…